# 언론사 크롤링 가능성 테스트

Google News RSS로 검색한 기사의 본문을 정확히 가져올 수 있는 언론사를 추리는 노트북입니다. 위에서부터 순서대로 실행하면 됩니다.

**단계**
1. Google News RSS 검색 (검색어 × 언론사 × 날짜)
2. Google News 링크 → 언론사 원문 URL 복원
3. 언론사별 샘플 추출 → 원문 HTML 다운로드
4. 저장된 HTML로 본문 추출 + 페이지 판정 (영상 클립 / 라이브 블로그 / 페이월 / 차단 / 본문 부족 …)
5. 언론사별 요약 → 판정별 언론사 딕셔너리 생성

**캐시**: 1~3단계 결과는 모두 `source_test/` 폴더에 저장됩니다.
다시 실행하면 이미 받은 RSS / URL / HTML은 재요청하지 않고 파일에서 읽어옵니다.
4단계 판정 기준(임계값, 패턴)만 바꿔서 다시 돌릴 때는 네트워크 요청이 전혀 발생하지 않습니다.

In [1]:
import gzip
import hashlib
import json
import random
import re
import time
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import quote_plus, urlparse
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
from googlenewsdecoder import gnews_decoder_async
from lxml import html as lxml_html
from trafilatura import extract
from trafilatura.utils import decode_file

## 0. 설정

In [2]:
# ============================================================
# 테스트 대상
# ============================================================

CANDIDATE_SOURCES = {
    # 미국 증시/금융 전문
    "finance.yahoo.com": "Yahoo Finance",
    "cnbc.com": "CNBC",
    "marketwatch.com": "MarketWatch",
    "barrons.com": "Barron's",

    # 경제/비즈니스 종합
    "fortune.com": "Fortune",
    "forbes.com": "Forbes",
    "businessinsider.com": "Business Insider",
    "cnn.com": "CNN Business",
    "foxbusiness.com": "Fox Business",
    "apnews.com": "Associated Press",

    # 투자 전문
    "benzinga.com": "Benzinga",
    "thestreet.com": "TheStreet",
    "fool.com": "The Motley Fool",
    "seekingalpha.com": "Seeking Alpha",
    "investing.com": "Investing.com",
    "zacks.com": "Zacks",

    # 접근 제한 가능성이 높은 곳
    "bloomberg.com": "Bloomberg",
    "wsj.com": "Wall Street Journal",
    "ft.com": "Financial Times",
    "reuters.com": "Reuters",

    # 추가 후보
    "nasdaq.com": "Nasdaq",
    "investopedia.com": "Investopedia",
    "kiplinger.com": "Kiplinger",
    "axios.com": "Axios",
    "cbsnews.com": "CBS News",
    "nypost.com": "New York Post",
    "usatoday.com": "USA Today",
    "theguardian.com": "The Guardian",
}

TEST_QUERIES = [
    "stock market",
    "Wall Street",
    "Federal Reserve",
    "earnings",
    "NVIDIA",
]

# 연속 구간 대신 서로 떨어진 날짜를 골라 표본 다양성 확보
TEST_DATES = [
    "2026-09-29",
    "2026-10-14",
    "2026-10-21",
]

NEWS_TIMEZONE = ZoneInfo("America/New_York")

# ============================================================
# 캐시 경로
# ============================================================

CACHE_DIR = Path("source_test")
HTML_DIR = CACHE_DIR / "html"

RSS_CACHE_PATH = CACHE_DIR / "rss_cache.jsonl"
RESOLVE_CACHE_PATH = CACHE_DIR / "resolve_cache.jsonl"
FETCH_CACHE_PATH = CACHE_DIR / "fetch_cache.jsonl"

PAGE_RESULT_PATH = CACHE_DIR / "page_results.parquet"
SOURCE_SUMMARY_PATH = CACHE_DIR / "source_summary.csv"
SOURCE_LISTS_PATH = CACHE_DIR / "source_lists.json"
REVIEW_SAMPLE_PATH = CACHE_DIR / "review_samples.csv"

# ============================================================
# 요청 설정
# ============================================================

REQUEST_TIMEOUT = 15
RSS_SLEEP_SEC = 1.0
FETCH_SLEEP_SEC = 0.5

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

# ============================================================
# 샘플링 / 판정 기준
# ============================================================

SAMPLE_PER_SOURCE = 30      # 언론사당 크롤링할 최대 기사 수
SAMPLE_SEED = 42

MIN_BODY_WORDS = 80         # 본문 최소 단어 수 (짧은 단신 기사도 통과하도록 낮게)
PARAGRAPH_MIN_WORDS = 15    # 참고용 문단 수 계산 시, 이 단어 수 이상인 줄만 문단으로 인정
MIN_LD_COVERAGE = 0.6       # JSON-LD articleBody 대비 추출 본문 비율 하한

MIN_ARTICLE_PAGES = 10      # 판정에 필요한 최소 기사 페이지 수
USABLE_RATE = 0.8
UNSTABLE_RATE = 0.5
VIDEO_HEAVY_SHARE = 0.3     # 영상/라이브/비기사 페이지 비율이 이 이상이면 별도 표시

# ============================================================
# 페이지 유형 판별 패턴
# ============================================================

# URL 경로에 이 구간이 있으면 영상/오디오/갤러리 페이지
VIDEO_PATH_PATTERN = re.compile(
    r"/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)",
    re.IGNORECASE,
)

# 호스트가 video.xxx.com 형태인 경우
VIDEO_HOST_PATTERN = re.compile(r"^(video|videos|watch|tv)\.", re.IGNORECASE)

LIVE_BLOG_PATH_PATTERN = re.compile(
    r"/(live|live-news|live-updates|liveblog)(/|$)|live-updates",
    re.IGNORECASE,
)

ARTICLE_LD_TYPES = {
    "Article",
    "NewsArticle",
    "ReportageNewsArticle",
    "AnalysisNewsArticle",
    "OpinionNewsArticle",
    "BackgroundNewsArticle",
    "ReviewNewsArticle",
    "BlogPosting",
    "Report",
}

PAYWALL_PHRASES = [
    "subscribe to continue",
    "subscribe to read",
    "subscription required",
    "to continue reading",
    "already a subscriber",
    "sign in to continue",
    "sign in to read",
    "log in to continue",
    "create a free account to continue",
    "this article is for subscribers",
    "this content is for subscribers",
    "unlock this article",
    "become a member to read",
]

# 봇 차단 / 챌린지 페이지 흔적 (<title> 또는 HTML 원문에서 검사)
BLOCK_TITLE_PHRASES = [
    "access denied",
    "just a moment",
    "attention required",
    "are you a robot",
    "pardon our interruption",
    "request unsuccessful",
    "verify you are human",
    "403 forbidden",
]

BLOCK_HTML_MARKERS = [
    "px-captcha",
    "captcha-delivery.com",
    "cf-chl-",
    "_incapsula_resource",
]

VERDICT_ORDER = ["USABLE", "UNSTABLE", "NOT_USABLE", "INSUFFICIENT_SAMPLE", "NO_RSS_RESULT"]

# 영상·라이브·비기사(작성자 프로필 등) 페이지는 URL/메타데이터 규칙으로
# 걸러낼 수 있으므로 성공률 계산 시 분모에서 제외
FILTERABLE_STATUSES = {"VIDEO_PAGE", "LIVE_BLOG", "NOT_ARTICLE"}

## 0-1. 공통 유틸 (캐시 읽기/쓰기)

In [3]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)

# 다운로드한 HTML이 git에 올라가지 않도록 캐시 폴더 전체를 제외
(CACHE_DIR / ".gitignore").write_text("*\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def url_hash(url):
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def html_cache_path(url):
    return HTML_DIR / f"{url_hash(url)}.html.gz"


def host_of(url):
    host = (urlparse(url).hostname or "").lower()

    if host.startswith("www."):
        host = host[4:]

    return host


def host_matches_source(url, source):
    host = host_of(url)

    return host == source or host.endswith("." + source)


def to_source_dict(domains):
    """도메인 리스트 → {도메인: 언론사명} (CANDIDATE_SOURCES 순서 유지)"""
    domains = set(domains)

    return {
        domain: name
        for domain, name in CANDIDATE_SOURCES.items()
        if domain in domains
    }


def make_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    return session

## 1. Google News RSS 검색 (캐시)

In [4]:
def build_rss_url(query, source, target_date):
    """
    target_date 앞뒤 하루를 포함해 검색하고
    실제 날짜 필터링은 Python에서 미국 동부시간 기준으로 다시 수행합니다.
    """
    search_after = target_date - timedelta(days=1)
    search_before = target_date + timedelta(days=1)

    search_query = (
        f'"{query}" '
        f"site:{source} "
        f"after:{search_after.isoformat()} "
        f"before:{search_before.isoformat()}"
    )

    url = (
        "https://news.google.com/rss/search"
        f"?q={quote_plus(search_query)}"
        "&hl=en-US"
        "&gl=US"
        "&ceid=US:en"
    )

    return url, search_query


def fetch_rss(query, source, target_date, session):
    url, search_query = build_rss_url(query, source, target_date)

    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()

    feed = feedparser.parse(response.content)

    entries = []

    for entry in feed.entries:
        published_raw = entry.get("published")

        if not published_raw:
            continue

        try:
            published_dt = parsedate_to_datetime(published_raw)
            published_et = published_dt.astimezone(NEWS_TIMEZONE)

        except Exception:
            continue

        if published_et.date() != target_date:
            continue

        rss_source = entry.get("source") or {}

        entries.append({
            "title": entry.get("title"),
            "google_news_url": entry.get("link"),
            "rss_source_title": rss_source.get("title"),
            "rss_source_href": rss_source.get("href"),
            "published_at": published_dt.isoformat(),
            "published_at_et": published_et.isoformat(),
        })

    return {
        "search_query": search_query,
        # 날짜 필터 전 RSS 항목 수 (100이면 결과가 잘렸을 가능성)
        "total_entries": len(feed.entries),
        "entries": entries,
    }


def rss_cache_key(query, source, target_date):
    return f"{target_date}|{source}|{query}"


def collect_rss_cached(sources, queries, dates, retry_errors=True):
    cache = {}

    for record in load_jsonl(RSS_CACHE_PATH):
        cache[record["key"]] = record  # 같은 key는 마지막 기록이 우선

    combos = [
        (datetime.strptime(d, "%Y-%m-%d").date(), source, query)
        for d in dates
        for source in sources
        for query in queries
    ]

    todo = []

    for target_date, source, query in combos:
        cached = cache.get(rss_cache_key(query, source, target_date))

        if cached and (cached["status"] == "OK" or not retry_errors):
            continue

        todo.append((target_date, source, query))

    print(f"RSS 조합 {len(combos)}개 중 캐시 사용 {len(combos) - len(todo)}개, 새로 요청 {len(todo)}개")

    session = make_session()

    for i, (target_date, source, query) in enumerate(todo, start=1):
        key = rss_cache_key(query, source, target_date)

        record = {
            "key": key,
            "target_date": target_date.isoformat(),
            "source": source,
            "query": query,
            "fetched_at": now_utc(),
        }

        try:
            result = fetch_rss(query, source, target_date, session)
            record.update(status="OK", error=None, **result)

        except Exception as e:
            record.update(
                status="ERROR",
                error=f"{type(e).__name__}: {e}",
                search_query=None,
                total_entries=0,
                entries=[],
            )

        append_jsonl(RSS_CACHE_PATH, record)
        cache[key] = record

        print(
            f"[{i}/{len(todo)}] {target_date} | {source} | {query} "
            f"→ {record['status']} {len(record['entries'])}건"
        )

        time.sleep(RSS_SLEEP_SEC)

    session.close()

    return [
        cache[rss_cache_key(query, source, target_date)]
        for target_date, source, query in combos
    ]

In [5]:
rss_records = collect_rss_cached(
    sources=CANDIDATE_SOURCES,
    queries=TEST_QUERIES,
    dates=TEST_DATES,
)

# 검색 결과 1건 = 1행 (같은 기사가 여러 검색어에 걸리면 여러 행)
rss_df = pd.DataFrame([
    {
        "target_date": record["target_date"],
        "source": record["source"],
        "query": record["query"],
        **entry,
    }
    for record in rss_records
    for entry in record["entries"]
])

# google_news_url 기준 중복 제거, 걸린 검색어는 리스트로 보존
articles_df = (
    rss_df
    .groupby(["source", "google_news_url"], as_index=False)
    .agg(
        query=("query", "first"),
        matched_queries=("query", lambda x: ", ".join(sorted(set(x)))),
        title=("title", "first"),
        rss_source_title=("rss_source_title", "first"),
        published_at_et=("published_at_et", "first"),
        target_date=("target_date", "first"),
    )
)

print(f"RSS 검색 결과 {len(rss_df)}행 → 중복 제거 {len(articles_df)}건")

RSS 조합 420개 중 캐시 사용 420개, 새로 요청 0개
RSS 검색 결과 7333행 → 중복 제거 5740건


In [6]:
rss_stats_df = (
    pd.DataFrame(rss_records)
    .assign(
        hits=lambda d: d["entries"].str.len(),
        is_error=lambda d: d["status"] == "ERROR",
        is_truncated=lambda d: d["total_entries"] >= 100,
    )
    .groupby("source")
    .agg(
        rss_requests=("key", "size"),
        rss_errors=("is_error", "sum"),
        rss_truncated=("is_truncated", "sum"),
        rss_hits=("hits", "sum"),
    )
    .join(articles_df.groupby("source").size().rename("rss_articles"))
    .fillna({"rss_articles": 0})
    .astype(int)
    .reindex(list(CANDIDATE_SOURCES))
    .reset_index()
)

# ── 변수 정리: RSS 단계 ──
RSS_HIT_SOURCES = to_source_dict(
    rss_stats_df.loc[rss_stats_df["rss_articles"] > 0, "source"]
)
RSS_EMPTY_SOURCES = to_source_dict(
    rss_stats_df.loc[rss_stats_df["rss_articles"] == 0, "source"]
)
RSS_ERROR_KEYS = [
    record["key"]
    for record in rss_records
    if record["status"] == "ERROR"
]

print("RSS 결과 있음:", list(RSS_HIT_SOURCES.values()))
print("RSS 결과 없음:", list(RSS_EMPTY_SOURCES.values()))
print("RSS 요청 실패:", len(RSS_ERROR_KEYS), "건")

rss_stats_df

RSS 결과 있음: ['Yahoo Finance', 'CNBC', 'MarketWatch', "Barron's", 'Fortune', 'Forbes', 'Business Insider', 'CNN Business', 'Fox Business', 'Associated Press', 'Benzinga', 'TheStreet', 'The Motley Fool', 'Seeking Alpha', 'Investing.com', 'Zacks', 'Bloomberg', 'Wall Street Journal', 'Financial Times', 'Reuters', 'Nasdaq', 'Investopedia', 'Kiplinger', 'Axios', 'CBS News', 'New York Post', 'USA Today', 'The Guardian']
RSS 결과 없음: []
RSS 요청 실패: 0 건


,source,rss_requests,rss_errors,rss_truncated,rss_hits,rss_articles
0,finance.yahoo.com,15,0,7,516,480
1,cnbc.com,15,0,2,256,192
2,marketwatch.com,15,0,2,220,176
3,barrons.com,15,0,3,341,210
4,fortune.com,15,0,0,220,170
5,forbes.com,15,0,0,209,186
6,businessinsider.com,15,0,5,367,313
7,cnn.com,15,0,0,91,69
8,foxbusiness.com,15,0,0,93,68
9,apnews.com,15,0,4,255,219


## 2. 원문 URL 복원 (캐시)

In [7]:
async def resolve_urls_cached(
    google_urls,
    batch_size=40,
    concurrency=4,
    retry_failed=False,
):
    cache = {}

    for record in load_jsonl(RESOLVE_CACHE_PATH):
        cache[record["google_news_url"]] = record

    unique_urls = list(dict.fromkeys(google_urls))

    todo = [
        url
        for url in unique_urls
        if url not in cache
        or (retry_failed and cache[url]["status"] != "SUCCESS")
    ]

    print(f"URL {len(unique_urls)}개 중 캐시 사용 {len(unique_urls) - len(todo)}개, 새로 복원 {len(todo)}개")

    for start in range(0, len(todo), batch_size):
        batch = todo[start:start + batch_size]

        try:
            results = await gnews_decoder_async(
                batch,
                interval=0.2,
                timeout=15.0,
                concurrency=concurrency,
            )

            if isinstance(results, dict):
                results = [results]

        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            results = [{"success": False, "message": error}] * len(batch)

        for url, result in zip(batch, results):
            record = {
                "google_news_url": url,
                "original_url": result.get("decoded_url") if result.get("success") else None,
                "status": "SUCCESS" if result.get("success") else "FAILED",
                "error": None if result.get("success") else result.get("message"),
                "resolved_at": now_utc(),
            }

            append_jsonl(RESOLVE_CACHE_PATH, record)
            cache[url] = record

        print(f"  {min(start + batch_size, len(todo))}/{len(todo)} 완료")

    return {url: cache[url] for url in unique_urls}

In [8]:
resolve_map = await resolve_urls_cached(
    articles_df["google_news_url"].tolist(),
)

articles_df["original_url"] = articles_df["google_news_url"].map(
    lambda url: resolve_map[url]["original_url"]
)
articles_df["resolve_status"] = articles_df["google_news_url"].map(
    lambda url: resolve_map[url]["status"]
)

# 복원된 URL이 검색한 언론사 도메인인지 (finance.yahoo.com 검색 → 다른 사이트로 연결되는 경우 체크)
articles_df["domain_match"] = [
    bool(url) and host_matches_source(url, source)
    for url, source in zip(articles_df["original_url"], articles_df["source"])
]

articles_df.groupby("source").agg(
    articles=("google_news_url", "size"),
    resolved=("resolve_status", lambda x: (x == "SUCCESS").sum()),
    domain_match=("domain_match", "sum"),
)

URL 5740개 중 캐시 사용 5740개, 새로 복원 0개


,articles,resolved,domain_match
source,,,
apnews.com,219,219,219
axios.com,30,30,30
barrons.com,210,210,210
benzinga.com,292,292,292
bloomberg.com,342,342,342
businessinsider.com,313,313,313
cbsnews.com,23,23,23
cnbc.com,192,192,192
cnn.com,69,69,69


## 3. 샘플 추출 + 원문 HTML 다운로드 (캐시)

In [9]:
def sample_for_crawl(articles_df, per_source=SAMPLE_PER_SOURCE, seed=SAMPLE_SEED):
    """
    언론사별로 검색어를 번갈아 가며 최대 per_source개를 뽑습니다.
    (특정 검색어 결과만 몰리지 않도록 라운드로빈)
    """
    candidates = (
        articles_df[
            (articles_df["resolve_status"] == "SUCCESS")
            & articles_df["domain_match"]
        ]
        .drop_duplicates("original_url")
        .sort_values(["source", "query", "google_news_url"])
    )

    sampled = []

    for source, group in candidates.groupby("source", sort=False):
        # 언론사마다 별도 시드 사용: 한 언론사에 기사가 추가돼도
        # 다른 언론사의 샘플은 그대로 유지되어 HTML을 다시 받지 않음
        rng = random.Random(f"{seed}|{source}")

        by_query = []

        for _, query_group in group.groupby("query"):
            rows = query_group.to_dict("records")
            rng.shuffle(rows)
            by_query.append(rows)

        picked = []

        while len(picked) < per_source and any(by_query):
            for rows in by_query:
                if rows and len(picked) < per_source:
                    picked.append(rows.pop())

        sampled.extend(picked)

    return pd.DataFrame(sampled)


def interleave_by_source(sample_df):
    """같은 사이트에 연속 요청하지 않도록 언론사를 번갈아 정렬"""
    queues = [
        group["original_url"].tolist()
        for _, group in sample_df.groupby("source")
    ]

    ordered = []

    while any(queues):
        for queue in queues:
            if queue:
                ordered.append(queue.pop(0))

    return ordered


def is_retryable_fetch(record):
    """일시적 오류(타임아웃, 429, 5xx)만 재시도 대상"""
    if record["fetch_status"] in {"TIMEOUT", "REQUEST_ERROR"}:
        return True

    return record["http_status"] in {429, 500, 502, 503, 504}


def fetch_html_cached(urls, retry_errors=False):
    cache = {}

    for record in load_jsonl(FETCH_CACHE_PATH):
        cache[record["url"]] = record

    todo = [
        url
        for url in urls
        if url not in cache
        or (retry_errors and is_retryable_fetch(cache[url]))
    ]

    print(f"URL {len(urls)}개 중 캐시 사용 {len(urls) - len(todo)}개, 새로 다운로드 {len(todo)}개")

    session = make_session()

    for i, url in enumerate(todo, start=1):
        record = {
            "url": url,
            "final_url": None,
            "http_status": None,
            "content_type": None,
            "fetch_status": None,
            "error": None,
            "html_saved": False,
            "fetched_at": now_utc(),
        }

        try:
            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT,
                allow_redirects=True,
            )

        except requests.Timeout:
            record.update(fetch_status="TIMEOUT", error="Request timeout")

        except requests.RequestException as e:
            record.update(fetch_status="REQUEST_ERROR", error=f"{type(e).__name__}: {e}")

        else:
            content_type = response.headers.get("Content-Type", "").lower()

            record.update(
                final_url=response.url,
                http_status=response.status_code,
                content_type=content_type,
            )

            if response.status_code != 200:
                record["fetch_status"] = "HTTP_ERROR"
            elif "html" not in content_type:
                record["fetch_status"] = "NOT_HTML"
            else:
                record["fetch_status"] = "OK"

            # 오류 응답이라도 HTML이면 저장 (차단 페이지 확인용)
            if "html" in content_type:
                html_cache_path(url).write_bytes(gzip.compress(response.content))
                record["html_saved"] = True

        append_jsonl(FETCH_CACHE_PATH, record)
        cache[url] = record

        print(
            f"[{i}/{len(todo)}] {host_of(url)} | "
            f"{record['fetch_status']} ({record['http_status']})"
        )

        time.sleep(FETCH_SLEEP_SEC)

    session.close()

    return {url: cache[url] for url in urls}

In [10]:
sample_df = sample_for_crawl(articles_df)

print(f"샘플 기사 수: {len(sample_df)}")
print(sample_df.groupby("source").size().to_string())

샘플 기사 수: 833
source
apnews.com             30
axios.com              30
barrons.com            30
benzinga.com           30
bloomberg.com          30
businessinsider.com    30
cbsnews.com            23
cnbc.com               30
cnn.com                30
finance.yahoo.com      30
fool.com               30
forbes.com             30
fortune.com            30
foxbusiness.com        30
ft.com                 30
investing.com          30
investopedia.com       30
kiplinger.com          30
marketwatch.com        30
nasdaq.com             30
nypost.com             30
reuters.com            30
seekingalpha.com       30
theguardian.com        30
thestreet.com          30
usatoday.com           30
wsj.com                30
zacks.com              30


In [11]:
fetch_map = fetch_html_cached(interleave_by_source(sample_df))

URL 833개 중 캐시 사용 833개, 새로 다운로드 0개


## 4. 본문 추출 + 페이지 판정 (저장된 HTML만 사용, 네트워크 요청 없음)

판정 기준(0. 설정의 임계값·패턴)을 바꾼 뒤 이 섹션부터 다시 실행하면 됩니다.

In [12]:
def read_cached_html(url):
    path = html_cache_path(url)

    if not path.exists():
        return None

    return decode_file(gzip.decompress(path.read_bytes()))


def parse_json_ld(tree):
    """페이지의 JSON-LD 최상위 객체(+@graph)를 평탄화해서 리스트로 반환"""
    items = []

    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            data = json.loads(node.text_content().strip())

        except Exception:
            continue

        stack = [data]

        while stack:
            current = stack.pop()

            if isinstance(current, list):
                stack.extend(current)

            elif isinstance(current, dict):
                items.append(current)

                if "@graph" in current:
                    stack.append(current["@graph"])

    return items


def ld_types_of(item):
    types = item.get("@type", [])

    if isinstance(types, str):
        types = [types]

    return {t for t in types if isinstance(t, str)}


def meta_content(tree, attr, value):
    found = tree.xpath(f'//meta[@{attr}="{value}"]/@content')

    return found[0].strip().lower() if found else None


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"

    return None


def extract_page_features(raw_html, final_url):
    # lxml은 encoding 선언이 있는 str을 받지 못하므로 XML 선언 제거
    tree = lxml_html.fromstring(re.sub(r"^\s*<\?xml[^>]*\?>", "", raw_html))

    page_title = (tree.findtext(".//title") or "").strip()

    # ── 메타데이터 ──
    og_type = meta_content(tree, "property", "og:type")
    twitter_card = meta_content(tree, "name", "twitter:card")
    has_og_video = bool(tree.xpath('//meta[starts-with(@property, "og:video")]'))

    ld_items = parse_json_ld(tree)
    ld_types = set().union(*[ld_types_of(item) for item in ld_items]) if ld_items else set()

    article_items = [item for item in ld_items if ld_types_of(item) & ARTICLE_LD_TYPES]

    ld_body_words = max(
        (len(str(item.get("articleBody") or "").split()) for item in article_items),
        default=0,
    )

    ld_free_values = [
        parse_bool(item.get("isAccessibleForFree"))
        for item in article_items
        if "isAccessibleForFree" in item
    ]
    ld_is_free = None if not ld_free_values else all(v is not False for v in ld_free_values)

    # ── URL 기반 ──
    path = urlparse(final_url).path

    url_is_video = bool(
        VIDEO_PATH_PATTERN.search(path)
        or VIDEO_HOST_PATTERN.search(host_of(final_url))
    )
    url_is_live = bool(LIVE_BLOG_PATH_PATTERN.search(path))

    # ── 차단 페이지 흔적 ──
    title_lower = page_title.lower()
    html_lower = raw_html[:50000].lower()

    is_blocked = (
        any(phrase in title_lower for phrase in BLOCK_TITLE_PHRASES)
        or any(marker in html_lower for marker in BLOCK_HTML_MARKERS)
    )

    # ── 본문 ──
    body = extract(
        raw_html,
        url=final_url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    body = (body or "").strip()

    body_words = len(body.split())
    body_paragraphs = sum(
        1
        for line in body.split("\n")
        if len(line.split()) >= PARAGRAPH_MIN_WORDS
    )

    body_lower = body.lower()
    paywall_phrase = next((p for p in PAYWALL_PHRASES if p in body_lower), None)

    return {
        "page_title": page_title,
        "og_type": og_type,
        "twitter_card": twitter_card,
        "has_og_video": has_og_video,
        "ld_types": sorted(ld_types),
        "ld_has_article": bool(ld_types & ARTICLE_LD_TYPES),
        "ld_has_video": "VideoObject" in ld_types,
        "ld_has_liveblog": "LiveBlogPosting" in ld_types,
        "ld_body_words": ld_body_words,
        "ld_is_free": ld_is_free,
        "url_is_video": url_is_video,
        "url_is_live": url_is_live,
        "is_blocked": is_blocked,
        "paywall_phrase": paywall_phrase,
        "body": body,
        "body_words": body_words,
        "body_paragraphs": body_paragraphs,
        "ld_coverage": (body_words / ld_body_words) if ld_body_words else None,
    }


def classify_page(source, fetch, features):
    if fetch is None:
        return "NOT_FETCHED"

    if fetch["fetch_status"] in {"TIMEOUT", "REQUEST_ERROR", "NOT_HTML"}:
        return fetch["fetch_status"]

    if features is not None and features["is_blocked"]:
        return "BLOCKED"

    if fetch["fetch_status"] == "HTTP_ERROR":
        return f"HTTP_{fetch['http_status']}"

    if not host_matches_source(fetch["final_url"], source):
        return "REDIRECTED_AWAY"

    if features is None:
        return "NOT_FETCHED"

    # 영상 클립: URL이 영상 경로이거나, 메타데이터상 기사가 아니라 영상인 페이지
    if (
        features["url_is_video"]
        or (features["og_type"] or "").startswith("video")
        or (features["ld_has_video"] and not features["ld_has_article"])
    ):
        return "VIDEO_PAGE"

    if features["url_is_live"] or features["ld_has_liveblog"]:
        return "LIVE_BLOG"

    # 작성자 프로필, 태그 목록 등 기사 메타데이터가 없는 페이지
    if not features["ld_has_article"] and features["og_type"] != "article":
        return "NOT_ARTICLE"

    if not features["body"]:
        return "EXTRACT_FAILED"

    if features["paywall_phrase"] or features["ld_is_free"] is False:
        return "PAYWALL"

    # 문단 수는 trafilatura 줄바꿈이 사이트마다 달라 판정에 쓰지 않음 (참고용 컬럼)
    if features["body_words"] < MIN_BODY_WORDS:
        return "TOO_SHORT"

    if (
        features["ld_coverage"] is not None
        and features["ld_coverage"] < MIN_LD_COVERAGE
    ):
        return "PARTIAL_BODY"

    return "SUCCESS"


def analyze_samples(sample_df, fetch_map):
    rows = []

    for article in sample_df.to_dict("records"):
        url = article["original_url"]
        fetch = fetch_map.get(url)

        raw_html = read_cached_html(url) if fetch and fetch["html_saved"] else None

        features = None
        analyze_error = None

        if raw_html:
            try:
                features = extract_page_features(raw_html, fetch["final_url"])

            except Exception as e:
                analyze_error = f"{type(e).__name__}: {e}"

        status = classify_page(article["source"], fetch, features)

        if analyze_error:
            status = "ANALYZE_ERROR"

        rows.append({
            "source": article["source"],
            "source_name": CANDIDATE_SOURCES.get(article["source"]),
            "query": article["query"],
            "title": article["title"],
            "published_at_et": article["published_at_et"],
            "original_url": url,
            "final_url": fetch["final_url"] if fetch else None,
            "http_status": fetch["http_status"] if fetch else None,
            "status": status,
            "analyze_error": analyze_error,
            **(features or {}),
        })

    return pd.DataFrame(rows)

In [13]:
page_df = analyze_samples(sample_df, fetch_map)
page_df.to_parquet(PAGE_RESULT_PATH, index=False)

print(f"저장: {PAGE_RESULT_PATH} ({len(page_df)}건)")

pd.crosstab(page_df["source"], page_df["status"], margins=True)

저장: source_test/page_results.parquet (833건)


status,BLOCKED,EXTRACT_FAILED,HTTP_401,HTTP_402,HTTP_403,LIVE_BLOG,NOT_ARTICLE,PAYWALL,SUCCESS,TOO_SHORT,VIDEO_PAGE,All
source,,,,,,,,,,,,
apnews.com,30,0,0,0,0,0,0,0,0,0,0,30
axios.com,30,0,0,0,0,0,0,0,0,0,0,30
barrons.com,30,0,0,0,0,0,0,0,0,0,0,30
benzinga.com,30,0,0,0,0,0,0,0,0,0,0,30
bloomberg.com,30,0,0,0,0,0,0,0,0,0,0,30
businessinsider.com,0,1,0,0,0,0,0,8,20,1,0,30
cbsnews.com,0,0,0,0,0,2,0,0,16,0,5,23
cnbc.com,0,0,0,0,0,0,0,7,15,0,8,30
cnn.com,1,0,0,0,0,0,21,8,0,0,0,30


## 5. 언론사별 요약 + 판정

In [14]:
def summarize_sources(page_df, rss_stats_df):
    counts = pd.crosstab(page_df["source"], page_df["status"])

    summary = (
        rss_stats_df
        .set_index("source")
        .join(page_df.groupby("source").size().rename("sampled"))
        .join(counts)
        .fillna(0)
    )

    status_cols = list(counts.columns)
    summary[status_cols + ["sampled"]] = summary[status_cols + ["sampled"]].astype(int)

    success = summary.get("SUCCESS", 0)
    filterable = sum(summary.get(s, 0) for s in FILTERABLE_STATUSES)

    summary["article_pages"] = summary["sampled"] - filterable
    summary["filterable_share"] = (filterable / summary["sampled"]).where(summary["sampled"] > 0)
    summary["success_rate"] = (success / summary["sampled"]).where(summary["sampled"] > 0)
    summary["article_success_rate"] = (success / summary["article_pages"]).where(summary["article_pages"] > 0)

    summary["median_words"] = (
        page_df[page_df["status"] == "SUCCESS"]
        .groupby("source")["body_words"]
        .median()
    )

    # 성공이 아닌 페이지 중 가장 많은 실패 사유
    failures = page_df[~page_df["status"].isin({"SUCCESS"} | FILTERABLE_STATUSES)]
    summary["main_fail_reason"] = failures.groupby("source")["status"].agg(
        lambda x: x.value_counts().index[0]
    )

    def verdict(row):
        if row["rss_articles"] == 0:
            return "NO_RSS_RESULT"

        if row["article_pages"] < MIN_ARTICLE_PAGES:
            return "INSUFFICIENT_SAMPLE"

        if row["article_success_rate"] >= USABLE_RATE:
            return "USABLE"

        if row["article_success_rate"] >= UNSTABLE_RATE:
            return "UNSTABLE"

        return "NOT_USABLE"

    summary["verdict"] = summary.apply(verdict, axis=1)
    summary.insert(0, "source_name", summary.index.map(CANDIDATE_SOURCES))

    summary["verdict"] = pd.Categorical(summary["verdict"], categories=VERDICT_ORDER, ordered=True)

    return (
        summary
        .reset_index()
        .sort_values(["verdict", "article_success_rate"], ascending=[True, False])
        .reset_index(drop=True)
    )

In [15]:
source_summary = summarize_sources(page_df, rss_stats_df)
source_summary.to_csv(SOURCE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

source_summary[[
    "source",
    "source_name",
    "verdict",
    "rss_articles",
    "sampled",
    "article_pages",
    "article_success_rate",
    "filterable_share",
    "median_words",
    "main_fail_reason",
]]

,source,source_name,verdict,rss_articles,sampled,article_pages,article_success_rate,filterable_share,median_words,main_fail_reason
0,finance.yahoo.com,Yahoo Finance,USABLE,480,30,28,1.000000,0.066667,407.0,NaN
1,foxbusiness.com,Fox Business,USABLE,68,30,14,1.000000,0.533333,480.5,NaN
2,fool.com,The Motley Fool,USABLE,348,30,30,1.000000,0.000000,680.5,NaN
3,cbsnews.com,CBS News,USABLE,23,23,16,1.000000,0.304348,657.0,NaN
4,nypost.com,New York Post,USABLE,59,30,30,1.000000,0.000000,592.0,NaN
5,theguardian.com,The Guardian,USABLE,73,30,23,1.000000,0.233333,836.0,NaN
6,usatoday.com,USA Today,USABLE,48,30,30,0.866667,0.000000,909.0,HTTP_402
7,cnbc.com,CNBC,UNSTABLE,192,30,22,0.681818,0.266667,495.0,PAYWALL
8,businessinsider.com,Business Insider,UNSTABLE,313,30,30,0.666667,0.000000,760.0,PAYWALL
9,fortune.com,Fortune,UNSTABLE,170,30,29,0.586207,0.033333,1193.0,PAYWALL


In [16]:
# ── 변수 정리: 최종 판정 ──
def sources_where(mask):
    return to_source_dict(source_summary.loc[mask, "source"])


USABLE_SOURCES = sources_where(source_summary["verdict"] == "USABLE")
UNSTABLE_SOURCES = sources_where(source_summary["verdict"] == "UNSTABLE")
NOT_USABLE_SOURCES = sources_where(source_summary["verdict"] == "NOT_USABLE")
INSUFFICIENT_SOURCES = sources_where(source_summary["verdict"] == "INSUFFICIENT_SAMPLE")
NO_RSS_SOURCES = sources_where(source_summary["verdict"] == "NO_RSS_RESULT")

# 영상/라이브 페이지 비중이 높은 언론사 (수집 시 URL 필터 필수)
VIDEO_HEAVY_SOURCES = sources_where(source_summary["filterable_share"] >= VIDEO_HEAVY_SHARE)

# 언론사별 주 실패 사유 {도메인: 상태값}
FAIL_REASON_BY_SOURCE = (
    source_summary
    .dropna(subset=["main_fail_reason"])
    .set_index("source")["main_fail_reason"]
    .to_dict()
)

source_lists = {
    "created_at": now_utc(),
    "test_queries": TEST_QUERIES,
    "test_dates": TEST_DATES,
    "USABLE": USABLE_SOURCES,
    "UNSTABLE": UNSTABLE_SOURCES,
    "NOT_USABLE": NOT_USABLE_SOURCES,
    "INSUFFICIENT_SAMPLE": INSUFFICIENT_SOURCES,
    "NO_RSS_RESULT": NO_RSS_SOURCES,
    "VIDEO_HEAVY": VIDEO_HEAVY_SOURCES,
    "FAIL_REASON": FAIL_REASON_BY_SOURCE,
}

SOURCE_LISTS_PATH.write_text(json.dumps(source_lists, ensure_ascii=False, indent=2))

for key in ["USABLE", "UNSTABLE", "NOT_USABLE", "INSUFFICIENT_SAMPLE", "NO_RSS_RESULT", "VIDEO_HEAVY"]:
    print(f"{key:20s}: {list(source_lists[key].values())}")

USABLE              : ['Yahoo Finance', 'Fox Business', 'The Motley Fool', 'CBS News', 'New York Post', 'USA Today', 'The Guardian']
UNSTABLE            : ['CNBC', 'Fortune', 'Business Insider']
NOT_USABLE          : ['MarketWatch', "Barron's", 'Forbes', 'Associated Press', 'Benzinga', 'TheStreet', 'Seeking Alpha', 'Investing.com', 'Zacks', 'Bloomberg', 'Wall Street Journal', 'Financial Times', 'Reuters', 'Investopedia', 'Kiplinger', 'Axios']
INSUFFICIENT_SAMPLE : ['CNN Business', 'Nasdaq']
NO_RSS_RESULT       : []
VIDEO_HEAVY         : ['CNN Business', 'Fox Business', 'Financial Times', 'Nasdaq', 'CBS News']


## 6. 눈으로 검수하기

판정이 실제와 맞는지 확인용입니다. 특히 `SUCCESS`인데 영상 클립 설명문만 있는 경우,
`TOO_SHORT`인데 실제로는 짧은 정상 기사인 경우를 확인하고 0. 설정의 기준을 조정하세요.

In [17]:
def show_samples(source, status=None, n=3, chars=800):
    rows = page_df[page_df["source"] == source]

    if status:
        rows = rows[rows["status"] == status]

    for row in rows.head(n).to_dict("records"):
        print(f"[{row['status']}] {row['title']}")
        print(f"URL: {row['final_url'] or row['original_url']}")
        print(
            f"단어 {row.get('body_words')} | 문단 {row.get('body_paragraphs')} | "
            f"og:type {row.get('og_type')} | LD {row.get('ld_types')}"
        )
        print(str(row.get("body") or "")[:chars])
        print("\n" + "=" * 80 + "\n")


# 언론사 × 상태별 최대 3건씩 검수용 CSV
review_df = (
    page_df
    .groupby(["source", "status"], group_keys=False)
    .head(3)
    .assign(body_head=lambda d: d["body"].fillna("").str[:500])
    [["source", "status", "title", "final_url", "body_words", "body_paragraphs", "og_type", "body_head"]]
    .sort_values(["source", "status"])
)
review_df.to_csv(REVIEW_SAMPLE_PATH, index=False, encoding="utf-8-sig")

print(f"저장: {REVIEW_SAMPLE_PATH} ({len(review_df)}건)")

저장: source_test/review_samples.csv (151건)


In [18]:
show_samples("cnbc.com", status="SUCCESS", n=2)

[SUCCESS] Bending Spoons skyrockets 41% as Berkshire Hathaway makes $330 million investment - CNBC
URL: https://www.cnbc.com/2026/10/14/berkshire-hathaway-investment-bending-spoons.html
단어 161.0 | 문단 5.0 | og:type article | LD ['NewsArticle']
Shares of Bending Spoons skyrocketed 41% after a new securities filing revealed that Berkshire Hathaway made a fresh $330 million investment into the company.
The latest infusion follows the conglomerate's $600 million investment from July, when the Italy-based serial acquirer went public on the Nasdaq.
Under the deal, Bending Spoons will issue 10 million ordinary shares to Berkshire at $33 a share.
"Bending Spoons continuously evaluates M&A opportunities, and our pipeline remains strong," said Francesco Patarnello, Bending Spoons' co-founder and head of business acquisitions, in a statement. "We believe Berkshire Hathaway's investment reflects confidence in our team and Platform and will further enhance our financial flexibility and ability to pu

In [19]:
show_samples("cnbc.com", status="VIDEO_PAGE", n=2)

[VIDEO_PAGE] Wamsi Mohan: Expect solid earnings season for Western Digital, Seagate, Dell - CNBC
URL: https://www.cnbc.com/video/2026/10/21/wamsi-mohan-expect-solid-earnings-season-for-western-digital-seagate-dell.html
단어 36.0 | 문단 1.0 | og:type video | LD ['VideoObject']
watch now
Wamsi Mohan: Expect solid earnings season for Western Digital, Seagate, Dell
Wamsi Mohan, Bank of American Securities senior IT hardware analyst, joins 'Power Lunch' to discuss earnings.
05:30
Wed, Oct 21 20262:20 PM EDT


[VIDEO_PAGE] 'Fast Money' traders react to Meta's Q3 results as stock climbs in extended trading - CNBC
URL: https://www.cnbc.com/video/2026/10/21/fast-money-traders-react-to-metas-q3-results-as-stock-climbs-in-extended-trading.html
단어 0.0 | 문단 0.0 | og:type video | LD ['VideoObject']





In [20]:
source_summary

,source,source_name,rss_requests,rss_errors,rss_truncated,rss_hits,rss_articles,sampled,BLOCKED,EXTRACT_FAILED,...,SUCCESS,TOO_SHORT,VIDEO_PAGE,article_pages,filterable_share,success_rate,article_success_rate,median_words,main_fail_reason,verdict
0,finance.yahoo.com,Yahoo Finance,15,0,7,516,480,30,0,0,...,28,0,1,28,0.066667,0.933333,1.000000,407.0,NaN,USABLE
1,foxbusiness.com,Fox Business,15,0,0,93,68,30,0,0,...,14,0,16,14,0.533333,0.466667,1.000000,480.5,NaN,USABLE
2,fool.com,The Motley Fool,15,0,8,500,348,30,0,0,...,30,0,0,30,0.000000,1.000000,1.000000,680.5,NaN,USABLE
3,cbsnews.com,CBS News,15,0,0,29,23,23,0,0,...,16,0,5,16,0.304348,0.695652,1.000000,657.0,NaN,USABLE
4,nypost.com,New York Post,15,0,0,67,59,30,0,0,...,30,0,0,30,0.000000,1.000000,1.000000,592.0,NaN,USABLE
5,theguardian.com,The Guardian,15,0,0,89,73,30,0,0,...,23,0,2,23,0.233333,0.766667,1.000000,836.0,NaN,USABLE
6,usatoday.com,USA Today,15,0,0,54,48,30,0,0,...,26,1,0,30,0.000000,0.866667,0.866667,909.0,HTTP_402,USABLE
7,cnbc.com,CNBC,15,0,2,256,192,30,0,0,...,15,0,8,22,0.266667,0.500000,0.681818,495.0,PAYWALL,UNSTABLE
8,businessinsider.com,Business Insider,15,0,5,367,313,30,0,1,...,20,1,0,30,0.000000,0.666667,0.666667,760.0,PAYWALL,UNSTABLE
9,fortune.com,Fortune,15,0,0,220,170,30,0,0,...,17,1,0,29,0.033333,0.566667,0.586207,1193.0,PAYWALL,UNSTABLE
